# YouTube Summarizer (Ollama)

This notebook fetches a YouTube transcript and summarizes it using a local Ollama model through the OpenAI-compatible endpoint. No pip commands are required if your environment already contains the dependencies.

## Notes

- Make sure Ollama is running locally: `ollama serve`
- Pull a model such as `llama3.2` if you have not already: `ollama pull llama3.2`
- If your environment already has `openai`, `youtube-transcript-api`, `tiktoken`, and `python-dotenv`, you can run the notebook directly.

In [ ]:
import os
import re
from dataclasses import dataclass
from typing import List

from openai import OpenAI
from youtube_transcript_api import YouTubeTranscriptApi
import tiktoken
from dotenv import load_dotenv

In [ ]:
from __future__ import annotations

@dataclass
class YouTubeVideoTranscript:
    url: str
    video_id: str
    title: str
    transcript_text: str


def load_environment():
    load_dotenv(override=True)


def get_ollama_base_url() -> str:
    return os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1")


def get_ollama_client() -> OpenAI:
    return OpenAI(api_key="ollama", base_url=get_ollama_base_url())


def extract_video_id(url: str) -> str:
    url = url.strip()
    patterns = [
        r"(?:https?://)?(?:www\.)?youtube\.com/watch\?v=([a-zA-Z0-9_-]+)",
        r"(?:https?://)?(?:www\.)?youtu\.be/([a-zA-Z0-9_-]+)",
        r"(?:https?://)?(?:www\.)?youtube\.com/embed/([a-zA-Z0-9_-]+)",
    ]

    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)

    raise ValueError("Unable to parse YouTube video id from URL")


def fetch_youtube_transcript(url: str) -> YouTubeVideoTranscript:
    video_id = extract_video_id(url)

    try:
        transcript_segments = YouTubeTranscriptApi().fetch(video_id)
    except Exception:
        print(f"INFO: No subtitles available for video ID: {video_id}.")
        transcript_text = ""
    else:
        if transcript_segments:
            transcript_text = " ".join([segment.text for segment in transcript_segments])
            print(f"Fetched transcript for video ID: {video_id}, length: {len(transcript_text)} characters")
        else:
            print(f"INFO: No subtitles available for video ID: {video_id}.")
            transcript_text = ""

    return YouTubeVideoTranscript(
        url=url,
        video_id=video_id,
        title=video_id,
        transcript_text=transcript_text,
    )

In [ ]:
def count_tokens(text: str, model: str = "gpt-4o-mini") -> int:
    try:
        encoding = tiktoken.encoding_for_model(model)
    except Exception:
        encoding = tiktoken.get_encoding("cl100k_base")
    return len(encoding.encode(text))


def split_text_into_chunks(text: str, max_tokens: int = 2500, overlap_tokens: int = 200) -> List[str]:
    if count_tokens(text) <= max_tokens:
        return [text]

    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks: List[str] = []
    current_chunk = ""

    for sentence in sentences:
        if not sentence.strip():
            continue

        candidate = f"{current_chunk} {sentence}".strip() if current_chunk else sentence
        if count_tokens(candidate) <= max_tokens:
            current_chunk = candidate
        else:
            if current_chunk:
                chunks.append(current_chunk)
            current_chunk = sentence
            if overlap_tokens and chunks:
                overlap_words = current_chunk.split()[: max(1, overlap_tokens // 3)]
                current_chunk = " ".join(overlap_words) + " " + sentence

    if current_chunk:
        chunks.append(current_chunk)

    return chunks

In [ ]:
def generate_system_prompt() -> str:
    return (
        "You are a helpful assistant that summarizes YouTube video transcripts. "
        "Produce a clear, structured summary with a short title, the main topic, and the key takeaways. "
        "When the transcript is long, focus on the most useful, high-value content and omit repeated fluff."
    )


def generate_user_prompt(title: str, transcript_chunk: str) -> str:
    return (
        f"Video Title: {title}\n\n"
        "Transcript:\n"
        f"{transcript_chunk}\n\n"
        "Summarize this transcript chunk into a concise, well-structured summary. "
        "Include a short bullet list of the top 3 insights or actions."
    )

In [ ]:
def summarize_chunk(client: OpenAI, title: str, transcript_chunk: str, model: str = "llama3.2") -> str:
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": generate_system_prompt()},
            {"role": "user", "content": generate_user_prompt(title, transcript_chunk)},
        ],
        max_tokens=800,
        temperature=0.2,
    )

    return response.choices[0].message.content.strip()


def summarize_transcript(transcript: YouTubeVideoTranscript, model: str = "llama3.2") -> str:
    client = get_ollama_client()
    chunks = split_text_into_chunks(transcript.transcript_text, max_tokens=2500, overlap_tokens=200)

    if len(chunks) == 1:
        return summarize_chunk(client, transcript.title, chunks[0], model=model)

    chunk_summaries: List[str] = []
    for index, chunk in enumerate(chunks, start=1):
        print(f"Summarizing chunk {index}/{len(chunks)}...")
        chunk_summaries.append(summarize_chunk(client, transcript.title, chunk, model=model))

    combined_prompt = (
        "You are an expert at combining several transcript summaries into one coherent final summary. "
        "Use the information below to create a single summary with a final short title, main topic, and key takeaways.\n\n"
        "Section summaries:\n"
        + "\n---\n".join(chunk_summaries)
    )

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You combine multiple summaries into one concise, high-quality summary."},
            {"role": "user", "content": combined_prompt},
        ],
        max_tokens=1000,
        temperature=0.2,
    )

    return response.choices[0].message.content.strip()

In [ ]:
# Load environment values and run the summarizer
load_environment()

# Set the YouTube URL below and execute this cell
youtube_url = "https://www.youtube.com/watch?v=l9JlsmzQs7Y"
transcript = fetch_youtube_transcript(youtube_url)
print(f"Video ID: {transcript.video_id}")
print(f"Transcript length: {len(transcript.transcript_text)} characters")

if transcript.transcript_text.strip():
    summary = summarize_transcript(transcript, model="llama3.2")
    print("\nFINAL SUMMARY")
    print(summary)
else:
    print("No transcript available; skipping summarization.")